In [9]:
import pandas as pd
import requests
import time
import numpy as np

In [10]:
PLAYLIST_CSV = "track_ids_private.csv"   # Your 40 song file
ARCHIVE_CSV = "tracks_features.csv"              # Your 1.2M song file (Kaggle)
OUTPUT_CSV = "final_playlist_features.csv"

In [11]:
print("LOADING DATA...")

# Load your playlist
df_playlist = pd.read_csv(PLAYLIST_CSV)
# Rename your column 'track_id' to 'spotify_id' to be consistent
df_playlist.rename(columns={'track_id': 'spotify_id'}, inplace=True)

# Load the archive (only keeping columns we need to save memory)
cols_to_keep = ['id', 'name', 'artists', 'danceability', 'energy', 'key', 
                'loudness', 'mode', 'speechiness', 'acousticness', 
                'instrumentalness', 'liveness', 'valence', 'tempo']
df_archive = pd.read_csv(ARCHIVE_CSV, usecols=lambda c: c in cols_to_keep)
df_archive.rename(columns={'id': 'spotify_id'}, inplace=True)

print(f"Playlist: {len(df_playlist)} songs")
print(f"Archive:  {len(df_archive)} songs")

LOADING DATA...
Playlist: 40 songs
Archive:  1204025 songs


In [12]:
print("\nSTARTING LOCAL MERGE...")

# A. Exact ID Match (The most accurate)
merged_df = pd.merge(df_playlist, df_archive, on='spotify_id', how='left', suffixes=('', '_archive'))

# Cleanup: If names overlap, keep the original playlist name
if 'name_archive' in merged_df.columns:
    merged_df.drop(columns=['name_archive', 'artists'], inplace=True, errors='ignore')

# B. Identify Missing Songs
missing_mask = merged_df['tempo'].isna()
df_missing = merged_df[missing_mask].copy()

print(f"Found via ID: {len(df_playlist) - len(df_missing)} songs")

# C. Name + Artist Rescue (For ID mismatches like 'Remasters')
if not df_missing.empty:
    print(f"Attempting to match {len(df_missing)} songs by Name/Artist...")
    
    # Create normalized columns for matching (lowercase, no spaces)
    df_missing['name_clean'] = df_missing['name'].astype(str).str.lower().str.strip()
    df_missing['artist_clean'] = df_missing['artist'].astype(str).str.lower().str.strip()
    
    df_archive['name_clean'] = df_archive['name'].astype(str).str.lower().str.strip()
    df_archive['artist_clean'] = df_archive['artists'].astype(str).str.lower().str.strip().str.replace(r"[\['\]']", "", regex=True)

    # Perform the "Rescue" Merge
    rescue_df = pd.merge(
        df_missing[['spotify_id', 'name_clean', 'artist_clean']], 
        df_archive, 
        on=['name_clean', 'artist_clean'], 
        how='inner'
    )
    
    # Remove duplicates
    rescue_df.drop_duplicates(subset=['spotify_id_x'], inplace=True)
    
    # Update the main DataFrame with rescued data
    if not rescue_df.empty:
        # We map the rescued values back to the main dataframe
        rescue_df.set_index('spotify_id_x', inplace=True)
        merged_df.set_index('spotify_id', inplace=True)
        merged_df.update(rescue_df)
        merged_df.reset_index(inplace=True)
        print(f"Rescued {len(rescue_df)} songs using Name Matching!")

# Re-check what is still missing
missing_mask = merged_df['tempo'].isna()
final_missing_ids = merged_df.loc[missing_mask, 'spotify_id'].unique().tolist()

print(f"------------------------------------------------")
print(f"Total Songs: {len(df_playlist)}")
print(f"Found Locally: {len(df_playlist) - len(final_missing_ids)}")
print(f"Still Missing: {len(final_missing_ids)} (Fetching from ReccoBeats...)")
print(f"------------------------------------------------")


STARTING LOCAL MERGE...
Found via ID: 19 songs
Attempting to match 21 songs by Name/Artist...
Rescued 2 songs using Name Matching!
------------------------------------------------
Total Songs: 40
Found Locally: 21
Still Missing: 19 (Fetching from ReccoBeats...)
------------------------------------------------


In [24]:
import urllib.parse

def query_acousticbrainz(artist, title):
    """
    Query AcousticBrainz High-Level features for a track.
    Returns a dictionary with features or None if not found.
    """
    artist_encoded = urllib.parse.quote_plus(artist)
    title_encoded = urllib.parse.quote_plus(title)
    
    url = f"https://acousticbrainz.org/api/v1/{artist_encoded}/{title_encoded}/high-level"
    try:
        response = requests.get(url, headers={'Accept': 'application/json'}, timeout=5)
        if response.status_code == 200:
            data = response.json()
            features = data.get("highlevel", {})
            return {
                "danceability": features.get("danceability", {}).get("value"),
                "energy": features.get("energy", {}).get("value"),
                "key": features.get("key_key", {}).get("value"),
                "loudness": features.get("average_loudness", {}).get("value"),
                "mode": features.get("key_scale", {}).get("value"),
                "speechiness": features.get("speechiness", {}).get("value"),
                "acousticness": features.get("acousticness", {}).get("value"),
                "instrumentalness": features.get("instrumentalness", {}).get("value"),
                "liveness": features.get("liveness", {}).get("value"),
                "valence": features.get("valence", {}).get("value"),
                "tempo": features.get("tempo", {}).get("value")
            }
        else:
            return None
    except Exception as e:
        print(f"⚠️ Error querying AcousticBrainz for {artist} - {title}: {e}")
        return None

In [25]:
if len(final_missing_ids) > 0:
    api_results = []
    print(f"Fetching audio features for {len(final_missing_ids)} missing tracks from AcousticBrainz...")

    for idx, row in df_missing.iterrows():
        artist = row['artist']
        title = row['name']
        print(f"[{idx+1}/{len(df_missing)}] Querying: {artist} - {title}")
        
        features = query_acousticbrainz(artist, title)
        if features:
            features["spotify_id"] = row["spotify_id"]
            api_results.append(features)
            print(f"✅ Retrieved features for {artist} - {title}")
        else:
            print(f"❌ Not found in AcousticBrainz: {artist} - {title}")
        
        time.sleep(1)  # be polite

    # Merge API results into main DataFrame
    if api_results:
        df_api = pd.DataFrame(api_results)
        merged_df.set_index('spotify_id', inplace=True)
        df_api.set_index('spotify_id', inplace=True)
        merged_df.update(df_api)
        merged_df.reset_index(inplace=True)
        print(f"\n✅ AcousticBrainz Merge Complete. Added features for {len(df_api)} tracks.")
    else:
        print("\n⚠️ No features retrieved from AcousticBrainz.")

Fetching audio features for 19 missing tracks from AcousticBrainz...
[1/21] Querying: Billie Eilish - L’AMOUR DE MA VIE
❌ Not found in AcousticBrainz: Billie Eilish - L’AMOUR DE MA VIE
[2/21] Querying: Chappell Roan - Good Luck, Babe!
❌ Not found in AcousticBrainz: Chappell Roan - Good Luck, Babe!
[3/21] Querying: Olivia Rodrigo - all-american bitch
❌ Not found in AcousticBrainz: Olivia Rodrigo - all-american bitch
[4/21] Querying: Billie Eilish - Oxytocin
❌ Not found in AcousticBrainz: Billie Eilish - Oxytocin
[7/21] Querying: G-Eazy - Lady Killers II
❌ Not found in AcousticBrainz: G-Eazy - Lady Killers II
[8/21] Querying: Kanye West, Chris Martin - Homecoming
❌ Not found in AcousticBrainz: Kanye West, Chris Martin - Homecoming
[9/21] Querying: Arctic Monkeys - Mardy Bum
❌ Not found in AcousticBrainz: Arctic Monkeys - Mardy Bum
[11/21] Querying: The Killers - Mr. Brightside
❌ Not found in AcousticBrainz: The Killers - Mr. Brightside
[15/21] Querying: Justice - D.A.N.C.E.
❌ Not found i

In [14]:
# Final check of data quality
final_count = merged_df['tempo'].count()
print(f"\nFinal Statistics:")
print(f"Total Songs: {len(merged_df)}")
print(f"Songs with Audio Features: {final_count}")

# Save
merged_df.to_csv(OUTPUT_CSV, index=False)
print(f"\nSUCCESS! Saved to {OUTPUT_CSV}")
print(merged_df[['name', 'tempo', 'valence', 'energy']].head(10))


Final Statistics:
Total Songs: 40
Songs with Audio Features: 21

SUCCESS! Saved to final_playlist_features.csv
                                       name    tempo  valence  energy
0                         L’AMOUR DE MA VIE      NaN      NaN     NaN
1                          Good Luck, Babe!      NaN      NaN     NaN
2                        all-american bitch      NaN      NaN     NaN
3                                  Oxytocin      NaN      NaN     NaN
4                    Fluorescent Adolescent  112.115    0.821   0.813
5                                      Kids  122.961    0.172   0.931
6                           Lady Killers II      NaN      NaN     NaN
7                                Homecoming      NaN      NaN     NaN
8                                 Mardy Bum  112.215    0.312   0.599
9  Why'd You Only Call Me When You're High?   92.004    0.800   0.631
